<a href="https://colab.research.google.com/github/viettungvuong/Distillation-Playground/blob/main/Distillation_Playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q peft bitsandbytes accelerate

**Huggingface login**

In [2]:
from huggingface_hub import login

login()

**Downloading dataset**

In [3]:
from datasets import load_dataset

dataset = load_dataset("OpenHust/vietnamese-summarization")

In [5]:
from transformers import AutoProcessor

model_name = "Qwen/Qwen3.6-27B"
processor = AutoProcessor.from_pretrained(model_name)

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig, AutoModelForCausalLM
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 5,242,880 || all params: 26,901,241,344 || trainable%: 0.0195


**Data processing**

In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Document', 'Summary', 'Dataset'],
        num_rows: 74564
    })
})

In [8]:
split_dataset = dataset["train"].train_test_split(test_size=0.2)

train_set = split_dataset['train']
test_set = split_dataset['test']

In [9]:
def normalize_text(row):
    # Only handle lowercase normalization
    row['Document'] = str(row['Document']).lower()
    row['Summary'] = str(row['Summary']).lower()
    return row

def add_special_tokens(row):
    # Qwen follows the ChatML format:
    # <|im_start|>user
    # {Document}<|im_end|>
    # <|im_start|>assistant
    # {Summary}<|im_end|>

    formatted_input = (
        f"<|im_start|>\n"
        f"{row['Document']}<|im_end|>\n"
        f"<|im_start|>\n"
        f"{row['Summary']}<|im_end|>"
    )

    row['Processed_Input'] = formatted_input
    return row

def process(row):
    normalized_row = normalize_text(row)
    return add_special_tokens(normalized_row)

train_set = train_set.map(process)
test_set = test_set.map(process)

Map:   0%|          | 0/59651 [00:00<?, ? examples/s]

Map:   0%|          | 0/14913 [00:00<?, ? examples/s]

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [11]:
print("Special tokens: ", tokenizer.special_tokens_map)
sep_token_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
print(f"Sep token is {sep_token_id}")

Special tokens:  {'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'vision_bos_token': '<|vision_start|>', 'vision_eos_token': '<|vision_end|>'}
Sep token is 248045


In [12]:
def tokenize_function(batch):
    # Tokenize the processed input
    model_inputs = tokenizer(
        batch["Processed_Input"],
        truncation=True,
        max_length=1542,
        padding='max_length'
    )

    labels = []

    for input_ids in model_inputs["input_ids"]:
        label = list(input_ids)
        try:
            # Find the index of the first separation token
            indices = [i for i, val in enumerate(label) if val == sep_token_id]
            turn_model_idx = indices[-1]
            # Set all tokens up to the first separation to -100 (loss function ignore)
            for i in range(turn_model_idx + 1):
                label[i] = -100
        except ValueError as e:
            print(e)

        labels.append(label)

    model_inputs["labels"] = labels
    return model_inputs


In [13]:
tokenized_train = train_set.map(tokenize_function, batched=True, remove_columns=train_set.column_names)
tokenized_test = test_set.map(tokenize_function, batched=True, remove_columns=test_set.column_names)

Map:   0%|          | 0/59651 [00:00<?, ? examples/s]

Map:   0%|          | 0/14913 [00:00<?, ? examples/s]

**Fine-tuning**

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="summarization",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    optim="paged_adamw_8bit"
)

In [16]:
from transformers import Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [17]:
# Re-initialize trainer to pick up the LoRA wrapped model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'bos_token_id': None, 'pad_token_id': 248044}.


Epoch,Training Loss,Validation Loss
